# Lending Club P2P Loan Default Prediction — Part 2: Improvements

Fixes three problems in the paper: data leakage, random split, and no gradient boosting.

## 1. Data and cleaning (reuse from baseline)

In [ ]:
# CELL 1 - Get the data (same as the baseline notebook)
import kagglehub, os, glob
data_dir = kagglehub.dataset_download("wordsforthewise/lending-club")
print("Data folder:", data_dir)

In [ ]:
# CELL 2 - Rebuild the paper's "data_clean.csv"
# The author's code reads a file called data_clean.csv that is NOT in the zip.
# The paper (Section III-A) says how it was made:
#   - loans issued 2012 to 2015
#   - only loans with a FINAL status: "Fully Paid", "Charged Off", "Default"
#     (so "Current", "Late", "In Grace Period" are removed)
import pandas as pd, numpy as np

csv_file = data_dir + "/accepted_2007_to_2018q4.csv/accepted_2007_to_2018Q4.csv"

# All columns the author's classification + regression code uses, plus issue_d to filter by year
author_cols = ['loan_amnt','funded_amnt','total_pymnt','term','int_rate','installment','issue_d','last_pymnt_d',
 'emp_length','home_ownership','annual_inc','verification_status','loan_status','purpose','zip_code','addr_state','dti',
 'delinq_2yrs','earliest_cr_line','fico_range_low','fico_range_high','inq_last_6mths','mths_since_last_delinq',
 'mths_since_last_record','open_acc','pub_rec','revol_bal','revol_util','total_acc','out_prncp','out_prncp_inv',
 'last_fico_range_high','last_fico_range_low','collections_12_mths_ex_med','mths_since_last_major_derog','policy_code',
 'application_type','annual_inc_joint','dti_joint','acc_now_delinq','tot_coll_amt','tot_cur_bal','open_acc_6m',
 'open_act_il','open_il_12m','open_il_24m','mths_since_rcnt_il','total_bal_il','il_util','open_rv_12m','open_rv_24m',
 'all_util','total_rev_hi_lim','inq_fi','total_cu_tl','inq_last_12m','acc_open_past_24mths','avg_cur_bal',
 'bc_open_to_buy','bc_util','chargeoff_within_12_mths','delinq_amnt','mo_sin_old_il_acct','mo_sin_old_rev_tl_op',
 'mo_sin_rcnt_rev_tl_op','mo_sin_rcnt_tl','mort_acc','mths_since_recent_bc','mths_since_recent_bc_dlq',
 'mths_since_recent_inq','mths_since_recent_revol_delinq','num_accts_ever_120_pd','num_actv_bc_tl','num_actv_rev_tl',
 'num_bc_sats','num_bc_tl','num_il_tl','num_op_rev_tl','num_rev_accts','num_rev_tl_bal_gt_0','num_sats',
 'num_tl_120dpd_2m','num_tl_30dpd','num_tl_90g_dpd_24m','num_tl_op_past_12m','pct_tl_nvr_dlq','percent_bc_gt_75',
 'pub_rec_bankruptcies','tax_liens','tot_hi_cred_lim','total_bal_ex_mort','total_bc_limit','total_il_high_credit_limit']

raw = pd.read_csv(csv_file, usecols=author_cols, low_memory=False)
print("Full Kaggle file:", raw.shape)

raw["issue_year"] = pd.to_datetime(raw["issue_d"], format="%b-%Y").dt.year
clean = raw[raw["issue_year"].between(2012, 2015) &
            raw["loan_status"].isin(["Fully Paid", "Charged Off", "Default"])].copy()
clean = clean.drop(columns=["issue_year"])
del raw

print("After filter (paper had 745,529):", len(clean))
print(clean["loan_status"].value_counts())
print((clean["loan_status"] != "Fully Paid").mean().round(3), "= share negative (paper: 0.19)")
clean.to_csv("data_clean.csv", index=False)

In [ ]:
# CELL 3 - Authors' cleaning rules (copied from baseline notebook)
import pandas as pd, numpy as np, gc

nan_max_cols = ['mths_since_last_delinq','mths_since_last_record','mths_since_last_major_derog','mths_since_rcnt_il',
 'mths_since_recent_bc','mths_since_recent_bc_dlq','mths_since_recent_inq','mths_since_recent_revol_delinq',
 'mo_sin_old_il_acct','mo_sin_old_rev_tl_op','mo_sin_rcnt_rev_tl_op','mo_sin_rcnt_tl']
nan_zero_cols = ['tot_coll_amt','tot_cur_bal','open_acc_6m','open_act_il','open_il_12m','open_il_24m','total_bal_il',
 'open_rv_12m','open_rv_24m','inq_fi','inq_last_12m','num_accts_ever_120_pd','num_actv_bc_tl','num_actv_rev_tl',
 'num_bc_sats','num_bc_tl','num_il_tl','num_op_rev_tl','num_rev_accts','num_rev_tl_bal_gt_0','num_sats',
 'num_tl_120dpd_2m','num_tl_30dpd','num_tl_90g_dpd_24m','num_tl_op_past_12m','mort_acc','percent_bc_gt_75',
 'tot_hi_cred_lim','total_bal_ex_mort','total_bc_limit','total_il_high_credit_limit','bc_open_to_buy',
 'acc_open_past_24mths','total_cu_tl','total_rev_hi_lim']
nan_mean_cols = ['dti','annual_inc_joint','dti_joint','il_util','all_util','bc_util','pct_tl_nvr_dlq','avg_cur_bal','revol_util']

def author_clean(df):
    df = df.copy()
    df['revol_util'] = pd.to_numeric(df['revol_util'].astype(str).str.replace('%',''), errors='coerce') / 100.0
    df['int_rate']   = pd.to_numeric(df['int_rate'].astype(str).str.replace('%',''), errors='coerce') / 100.0
    for col in df.columns:
        if col in nan_mean_cols: df[col] = df[col].fillna(np.nanmean(df[col].values))
        if col in nan_zero_cols: df[col] = df[col].fillna(0.0)
        if col in nan_max_cols:  df[col] = df[col].fillna(np.nanmax(df[col].values) * 5)
    d = pd.to_datetime(df['earliest_cr_line'], format='%b-%Y')
    df['earliest_cr_line'] = (d - pd.Timestamp('1970-01-01')).dt.days
    return df

## 2. Shared setup: three settings S1/S2/S3

In [ ]:
# CELL 4 - OUR IMPROVEMENTS - shared setup
# Everything above reproduces the paper. From here we change ONE thing at a time and measure the effect.
# This is called an ABLATION STUDY: it lets us say exactly which change caused which result.
#
# Three settings (what the model is allowed to see, and how we test it):
#   S1 = paper setup:   random 70/30 split, ALL features (including the leaky ones)
#   S2 = leak removed:  random 70/30 split, leaky columns dropped
#   S3 = honest test:   leaky columns dropped AND train on 2012-2014 loans, test on 2015 loans (the future)
#
# DATA LEAKAGE = giving the model information that only exists AFTER the answer is known:
#   last_fico_range_high / low : the borrower's credit score at the END of the loan (a defaulter's score crashes)
#   out_prncp / out_prncp_inv  : principal still owed - always 0 for a finished loan, so it carries no signal
#
# Part of the training data is held out as a VALIDATION set (15%). It is used only to tune / stop training,
# never for the final score - the paper had no validation set (its own Future Work section says so).
import gc, time, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_recall_fscore_support

LEAK_COLS = ['last_fico_range_high', 'last_fico_range_low', 'out_prncp', 'out_prncp_inv']

base = pd.read_csv("data_clean.csv", low_memory=False)
base['issue_year'] = pd.to_datetime(base['issue_d'], format='%b-%Y').dt.year
print("Share of defaults by issue year:")
print(base.groupby('issue_year')['loan_status'].apply(lambda s: (s != 'Fully Paid').mean()).round(3))
# Why 2015 is higher: the Kaggle file ends in Dec 2018, so a 5-year (60-month) loan from 2015 is only
# 'finished' if it was paid off early or defaulted. The 2015 test set therefore has more defaults.

def build_split(df, label_col, test_mask, val_frac=0.15, seed=0):
    # Same one-hot encoding + standardisation as the baseline, but the train/test split is GIVEN,
    # and a validation set is carved out of the training part.
    df = df.reset_index(drop=True)
    y = df[label_col].to_numpy(); test_mask = np.asarray(test_mask)
    df = df.drop(columns=[label_col])
    cat_cols = df.select_dtypes(include='object').columns.tolist()
    num_cols = [c for c in df.columns if c not in cat_cols]
    cats = {c: sorted(df[c].dropna().unique()) for c in cat_cols}
    names = num_cols + [c + '_' + str(v) for c in cat_cols for v in cats[c]]
    X = np.zeros((len(df), len(names)), dtype=np.float32)
    X[:, :len(num_cols)] = df[num_cols].to_numpy(dtype=np.float32)
    j = len(num_cols)
    for c in cat_cols:
        codes = pd.Categorical(df[c], categories=cats[c]).codes.astype(np.int64)
        rows = np.where(codes >= 0)[0]; X[rows, j + codes[rows]] = 1.0; j += len(cats[c])
    np.nan_to_num(X, copy=False)
    tr_idx = np.where(~test_mask)[0]; te_idx = np.where(test_mask)[0]
    rng = np.random.default_rng(seed); rng.shuffle(tr_idx)
    n_val = int(val_frac * len(tr_idx)); va_idx, tr_idx = tr_idx[:n_val], tr_idx[n_val:]
    m = X[tr_idx].mean(0, dtype=np.float64).astype(np.float32); s = X[tr_idx].std(0, dtype=np.float64).astype(np.float32); s[s == 0] = 1
    X -= m; X /= s
    return dict(Xtr=X[tr_idx], ytr=y[tr_idx], Xva=X[va_idx], yva=y[va_idx], Xte=X[te_idx], yte=y[te_idx], names=names)

def split_mask(name, year):
    if name == 'S3':
        return year == 2015                                      # test = future loans
    return np.random.default_rng(0).random(len(year)) < 0.3     # test = random 30%

def classification_setting(name):
    df = base.drop(columns=['total_pymnt', 'last_pymnt_d', 'issue_d']).copy()
    df['loan_status'] = df['loan_status'].map({'Charged Off': 0, 'Default': 0, 'Fully Paid': 1})
    year = df.pop('issue_year').to_numpy()
    if name in ('S2', 'S3'):
        df = df.drop(columns=LEAK_COLS)
    return build_split(author_clean(df), 'loan_status', split_mask(name, year))

def regression_setting(name):
    df = base.dropna(subset=['last_pymnt_d']).drop(columns=['loan_status']).copy()
    days = (pd.to_datetime(df['last_pymnt_d'], format='%b-%Y') - pd.to_datetime(df['issue_d'], format='%b-%Y')).dt.days
    df['label'] = (df['total_pymnt'] / df['funded_amnt']) ** (365.0 / (days + 30))   # author's label = 1 + NAR
    year = df.pop('issue_year').to_numpy()
    df = df.drop(columns=['total_pymnt', 'issue_d', 'last_pymnt_d'])
    if name in ('S2', 'S3'):
        df = df.drop(columns=LEAK_COLS)
    return build_split(author_clean(df), 'label', split_mask(name, year))